# Setup environment (for Google Colab)


In [ ]:
!apt-get update
!apt install chromium-chromedriver -y
!pip install selenium serpapi google-search-results
!pip install selenium webdriver-manager -q

In [ ]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!google-chrome --version

In [ ]:
!pip install webdriver-manager -q


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
print("Chrome works!")
driver.quit()

In [ ]:
import requests
import time
import json
import numpy as np
import re
from bs4 import BeautifulSoup
from serpapi.google_search import GoogleSearch
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

API_KEY = "PUT_API_CODE"
query = input("Enter course topic (e.g., Python): ")
courses = []

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")

path = ChromeDriverManager().install()
service = Service(path)
driver = webdriver.Chrome(service=service, options=options)

print("\n---Serpapi (Pluralsight)---\n")
search = GoogleSearch({
    "engine": "google",
    "api_key": API_KEY,
    "q": f"site:pluralsight.com/courses {query}",
    "hl": "en",
    "gl": "us",
    "num": 10
})

organic = search.get_dict().get("organic_results", [])
pluralsight_raw = []

for item in organic:
    link = item.get("link", "")
    if "pluralsight.com/courses/" not in link and "pluralsight.com/library/" not in link:
        continue
    raw_title = item.get("title", "N/A")
    title = raw_title.replace(" | Pluralsight", "").replace(" - Pluralsight", "").strip()
    clean_link = re.sub(r'\?.*', '', link)
    pluralsight_raw.append({"title": title, "link": clean_link})


HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

for course in pluralsight_raw:
    rating, price = "N/A", "N/A"
    duration_months = 0

    try:
        resp = requests.get(course["link"], headers=HEADERS, timeout=12)
        soup = BeautifulSoup(resp.text, "html.parser")

        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
                items = data if isinstance(data, list) else [data]
                for d in items:
                    if "aggregateRating" in d:
                        rv = d["aggregateRating"].get("ratingValue")
                        if rv:
                            rating = str(round(float(rv), 1))

                    # <--- EXTRACT DURATION (PLURALSIGHT MATH ENGINE) --->
                    if "timeRequired" in d:
                        tr = d["timeRequired"]
                        if "PT" in tr: # Pluralsight usually uses PT for Hours/Mins
                            duration_months = 1
                        elif "M" in tr:
                            match = re.search(r'(\d+)M', tr)
                            if match: duration_months = int(match.group(1))
                        elif "W" in tr:
                            match = re.search(r'(\d+)W', tr)
                            if match: duration_months = max(1, int(match.group(1)) // 4)
            except:
                pass

        if rating == "N/A":
            meta = soup.find("meta", {"name": re.compile(r'rating', re.I)})
            if meta and meta.get("content"):
                rating = meta["content"]

        # ── Rating fallback: visible stars text ─────────────────────────────
        if rating == "N/A":
            for tag in soup.find_all(["span", "div"]):
                txt = tag.get_text(strip=True)
                if re.match(r'^[1-5]\.\d$', txt):
                    rating = txt
                    break

        # <--- DURATION HTML FALLBACK --->
        if duration_months == 0:
            html_text = soup.get_text().lower()
            dur_match = re.search(r'(\d+)\s*(?:-|to)?\s*(\d+)?\s*months?', html_text)
            if dur_match:
                duration_months = int(dur_match.group(1))

        # ── Price: Pluralsight Math Engine (duration * 29) ──────────────────
        if duration_months > 0:
            price = str(duration_months * 29)
        else:
            price = "29" # Standard fallback (1 month) if duration is hidden

    except Exception as e:
        print(f"Failed: {course['link']} — {e}")

    platform = "Pluralsight"
    print(f"Title: {course['title']}")
    print(f"Link: {course['link']}")
    print(f"Rating: {rating}")
    print(f"Price: ${price} (Calculated: ~{max(1, duration_months)} Months)")
    print("-" * 60)
    courses.append({"title": course["title"], "link": course["link"], "platform": "Pluralsight", "source": "SerpAPI", "rating": rating, "price": price})


# ==============================================================================
# 2. COURSERA (BeautifulSoup + Custom Duration Math Engine)
# ==============================================================================
print("\n--- BeautifulSoup (Coursera) ---\n")

xml = requests.get("https://www.coursera.org/sitemap~www~courses.xml", headers={"User-Agent": "Mozilla/5.0"}).text
soup = BeautifulSoup(xml, "xml")
count = 0

for url in soup.find_all("loc"):
    link = url.text
    if "/learn/" in link and query.lower().replace(" ", "-") in link.lower():
        title = link.split("/")[-1].replace("-", " ").title()
        rating, price = "N/A", "N/A"
        duration_months = 0

        try:
            course_page = requests.get(link, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}).text
            course_soup = BeautifulSoup(course_page, "html.parser")

            for script in course_soup.find_all("script", type="application/ld+json"):
                try:
                    data = json.loads(script.string)
                    elements = data if isinstance(data, list) else [data]
                    for item in elements:
                        if "@graph" in item: elements.extend(item["@graph"])

                        if item.get("@type") == "Course" and "aggregateRating" in item:
                            rating = str(round(float(item["aggregateRating"]["ratingValue"]), 1))

                        if "timeRequired" in item:
                            tr = item["timeRequired"]
                            if "M" in tr:
                                match = re.search(r'(\d+)M', tr)
                                if match: duration_months = int(match.group(1))
                            elif "W" in tr:
                                match = re.search(r'(\d+)W', tr)
                                if match: duration_months = max(1, int(match.group(1)) // 4)
                except: pass

            if rating == "N/A":
                star_span = course_soup.find(attrs={"aria-label": re.compile(r'^[0-5]\.\d stars?$')})
                if star_span: rating = star_span['aria-label'].split(' ')[0]

            if duration_months == 0:
                html_text = course_soup.get_text().lower()
                dur_match = re.search(r'(\d+)\s*(?:-|to)?\s*(\d+)?\s*months?', html_text)
                if dur_match:
                    duration_months = int(dur_match.group(1))

            if duration_months > 0:
                price = str(duration_months * 20)
            else:
                price = "20"

        except Exception as e: pass

        if rating == "N/A": continue

        print(f"Title: {title}")
        print(f"Link: {link}")
        print(f"Rating: {rating}")
        print(f"Price: ${price} (Calculated: ~{max(1, duration_months)} Months)")
        print("-" * 50)

        courses.append({"title": title, "link": link, "platform": "Coursera", "source": "BeautifulSoup", "rating": rating, "price": price})
        count += 1
    if count >= 10: break

if count == 0: print("No Coursera courses with valid ratings found.")



# ==============================================================================
# 3. YOUTUBE (Selenium)
# ==============================================================================
print("\n--- Selenium (YouTube) ---\n")

youtube_url = f"https://www.youtube.com/results?search_query={query.replace(' ', '+')}+full+course&hl=en"
driver.get(youtube_url)
time.sleep(3)

count = 0
video_blocks = driver.find_elements(By.TAG_NAME, "ytd-video-renderer")[:20]

for video in video_blocks:
    try:
        title_elem = video.find_element(By.ID, "video-title")
        title = title_elem.get_attribute("title")
        href = title_elem.get_attribute("href")

        rating, price = "N/A", "0"

        match = re.search(r'([\d\.,]+[KM]?)\s*views', video.text, re.IGNORECASE)
        if match: rating = match.group(1) + " views"

        if rating == "N/A": continue

        if title and href:
            count += 1
            print(f"Title: {title}")
            print(f"Link: {href}")
            print(f"Views (Rating): {rating}")
            print(f"Price: Free")
            print("-" * 60)
            courses.append({"title": title, "link": href, "platform": "YouTube", "source": "Selenium", "rating": rating, "price": price})
            if count >= 10: break
    except Exception as e:
        continue

if count == 0: print("No YouTube courses found.")
driver.quit()


# ==============================================================================
# DATA CLEANING & EXPORT
# ==============================================================================
print("\n--- Cleaned Data ---")
df = pd.DataFrame(courses)
if not df.empty:
    df.drop_duplicates(subset=["title", "platform"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_csv("courses.csv", index=False)
    print(df.to_string())
else:
    print("No data collected to clean.")


# ==============================================================================
# NETWORK GRAPH
# ==============================================================================
print("\n netwerk")
PLATFORMS = {"coursera" , "pluralsight", "youtube"}
g=nx.Graph()
for _, row in df.iterrows():
  g.add_node(row["platform"] )
  g.add_node(row["title"])
  g.add_edge(row["platform"], row["title"])

print("\nAdjacency List")
for node in sorted(g.nodes()):
    if node.strip().lower() in PLATFORMS:
        print(f"{node} --> {sorted(list(g.neighbors(node)))}")
print("\nDegree of Each Vertex")
for node in sorted(g.nodes()):
    if node.strip().lower() in PLATFORMS:
      print(f"Vertex {node}: degree = {g.degree(node)}")
print("\nBetweenness Centrality(unnormalized)")
bc = nx.betweenness_centrality(g, normalized=False)
for node, centrality in sorted(bc.items()):
  if node.strip().lower() in PLATFORMS:
    print(f"Node {node}: {centrality}")
print("\nBetweenness Centrality(normalized)")
bc = nx.betweenness_centrality(g,)
for node, centrality in sorted(bc.items()):
  if node.strip().lower() in PLATFORMS:
    print(f"Node {node}: {centrality}")
print()
plt.figure(figsize=(22,16))

nx.draw(g,with_labels=True,node_size=800,width=2,font_size=6)
plt.title("platforms")
plt.show()


# ==============================================================================
# HEATMAP
# ==============================================================================
print("\n--- Distance-Based Heatmap (Rating vs Price) ---\n")

X_list, Y_list = [], []

for index, row in df.iterrows():
    r_str = str(row['rating']).lower().replace(" views", "").replace(",", "").strip()

    if "m" in r_str: r_num = float(r_str.replace("m", "")) * 1000000
    elif "k" in r_str: r_num = float(r_str.replace("k", "")) * 1000
    elif r_str != "n/a" and r_str != "nan":
        try: r_num = float(r_str)
        except: r_num = None
    else: r_num = None

    if r_num is not None and r_num > 10:
        r_num = min(5.0, max(1.0, 1 + (r_num / 500000) * 4))

    p_str = str(row['price']).lower().replace("$", "").replace("free", "0").replace(",", "").strip()
    # Handle subscription pricing
    if "subscription" in p_str: p_num = 29.0
    elif p_str != "n/a" and p_str != "nan" and "hidden" not in p_str:
        try: p_num = float(p_str.split()[0])
        except: p_num = None
    else: p_num = None

    if r_num is not None and p_num is not None:
        X_list.append(r_num)
        Y_list.append(p_num)

X, Y = np.array(X_list), np.array(Y_list)

if len(X) == 0:
    print("\nWarning: No valid numerical data extracted. Showing example data.")
    X, Y = np.array([4.5, 4.7, 4.2, 5.0, 4.8]), np.array([0.0, 29.0, 0.0, 0.0, 12.99])

grid_size = 100
x_min, x_max = 0, 6
y_min, y_max = -5, max(Y) + 20 if len(Y) > 0 else 20

x_grid, y_grid = np.linspace(x_min, x_max, grid_size), np.linspace(y_min, y_max, grid_size)
XX, YY = np.meshgrid(x_grid, y_grid)

heatmap = np.zeros((grid_size, grid_size))
R_x, R_y = 0.6, 10.0

for px, py in zip(X, Y):
    dist = np.sqrt(((XX - px) / R_x)**2 + ((YY - py) / R_y)**2)
    heatmap[dist <= 1.0] += 1

plt.figure(figsize=(10, 6))
plt.imshow(heatmap, extent=(x_min, x_max, y_min, y_max), origin="lower", cmap="hot", interpolation="nearest", aspect="auto")
plt.colorbar(label="Density")
plt.legend()
plt.title("Heatmap - Courses by Rating vs Price")
plt.xlabel("Rating (0-5 Stars / Scaled Views)")
plt.ylabel("Price ($)")
plt.show()